In [1]:
# ============================================================
# RANDOM SO(3) ROBUSTNESS BENCHMARK
#
# - random rotations via scipy Rotation.random()
# - rotation around image center
# - CT + GT masks rotated together
# - TotalSegmentator inference
# - Dice / IoU / Precision / Recall / Boundary IoU
# - memory-safe version
#
# ============================================================

# %pip install torchio scipy nibabel pandas TotalSegmentator

from pathlib import Path
import gc
import json
import math
import inspect
import contextlib
import os
import time
import copy
import numpy as np
import pandas as pd
import nibabel as nib
import torchio as tio

from scipy.spatial.transform import Rotation as R

from totalsegmentator.python_api import totalsegmentator


# ============================================================
# SETTINGS
# ============================================================

IMAGE_PATH = Path(
    r"C:\Users\semen\SEprojects\ct.nii\ct.nii"
)

MASK_DIR = Path(
    r"C:\Users\semen\SEprojects\segmentations\unzip_seg"
)

OUT_DIR = Path(
    r"C:\Users\semen\SEprojects\segmentations\so3_benchmark"
)

# number of random rotations
N_ROTATIONS = 300

RNG_SEED = 145

# instead of gigantic enclosing cube
PAD = 0

# crop back to original size
CROP_BACK = False

# ============================================================
# TotalSegmentator
# ============================================================

ROI_SUBSET = [
    "liver",
]

TS_TASK = "total"

TS_FAST = True

TS_QUIET = True

ROBUST_CROP = False

# ============================================================
# Metrics
# ============================================================

EMPTY_STRATEGY = "perfect"

BOUNDARY_RADIUS_VOX = 1


# ============================================================
# HELPERS
# ============================================================

def list_nii(folder: Path):

    out = []

    for p in sorted(folder.glob("*.nii")):
        if p.is_file():
            out.append(p)

    for p in sorted(folder.glob("*.nii.gz")):
        if p.is_file():
            out.append(p)

    return out


def out_name(p: Path):

    low = p.name.lower()

    if low.endswith(".nii.gz"):
        base = p.name[:-7]

    elif low.endswith(".nii"):
        base = p.name[:-4]

    else:
        base = p.stem

    # IMPORTANT:
    # .nii instead of .nii.gz
    return base + "_rot.nii"


def roi_key(name: str):

    low = name.lower()

    for suf in (".nii.gz", ".nii"):

        if low.endswith(suf):
            name = name[:-len(suf)]
            break

    if name.endswith("_rot"):
        name = name[:-4]

    return name.lower()


def load_mask_bool(path):

    nii = nib.load(str(path))

    arr = np.asarray(
        nii.dataobj,
        dtype=np.uint8
    )

    return arr > 0


# ============================================================
# Boundary IoU
# ============================================================

def shift_no_wrap(mask, axis, delta):

    out = np.zeros_like(mask, dtype=bool)

    src = [slice(None)] * 3
    dst = [slice(None)] * 3

    if delta == 1:

        src[axis] = slice(0, -1)
        dst[axis] = slice(1, None)

    elif delta == -1:

        src[axis] = slice(1, None)
        dst[axis] = slice(0, -1)

    out[tuple(dst)] = mask[tuple(src)]

    return out


def boundary_6n(mask):

    mask = mask.astype(bool)

    if mask.sum() == 0:
        return mask

    interior = mask.copy()

    for axis in range(3):

        interior &= shift_no_wrap(mask, axis, 1)
        interior &= shift_no_wrap(mask, axis, -1)

    return mask & (~interior)


def dilate_6n(mask, radius):

    out = mask.astype(bool)

    for _ in range(int(radius)):

        expanded = out.copy()

        for axis in range(3):

            expanded |= shift_no_wrap(out, axis, 1)
            expanded |= shift_no_wrap(out, axis, -1)

        out = expanded

    return out


def boundary_iou(gt, pr):

    gt = gt.astype(bool)
    pr = pr.astype(bool)

    b_gt = boundary_6n(gt)
    b_pr = boundary_6n(pr)

    if BOUNDARY_RADIUS_VOX > 0:

        b_gt = dilate_6n(
            b_gt,
            BOUNDARY_RADIUS_VOX
        )

        b_pr = dilate_6n(
            b_pr,
            BOUNDARY_RADIUS_VOX
        )

    inter = int(np.logical_and(b_gt, b_pr).sum())

    union = int(np.logical_or(b_gt, b_pr).sum())

    if union == 0:

        return (
            1.0
            if EMPTY_STRATEGY == "perfect"
            else float("nan")
        )

    return inter / union


def seg_metrics(gt, pr):

    gt = gt.astype(bool)
    pr = pr.astype(bool)

    tp = int(np.logical_and(gt, pr).sum())

    fp = int(np.logical_and(~gt, pr).sum())

    fn = int(np.logical_and(gt, ~pr).sum())

    gt_sum = int(gt.sum())

    pr_sum = int(pr.sum())

    if gt_sum + pr_sum == 0:

        if EMPTY_STRATEGY == "perfect":

            return {
                "dice": 1.0,
                "iou": 1.0,
                "precision": 1.0,
                "recall": 1.0,
                "boundary_iou": 1.0,
            }

        return {
            "dice": np.nan,
            "iou": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "boundary_iou": np.nan,
        }

    dice = (2 * tp) / (gt_sum + pr_sum)

    iou = tp / (tp + fp + fn)

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "boundary_iou": float(boundary_iou(gt, pr)),
    }


# ============================================================
# LOAD DATA
# ============================================================

img = tio.ScalarImage(IMAGE_PATH)

orig_shape = img.spatial_shape

subject_dict = {
    "img": img
}

mask_paths = list_nii(MASK_DIR)

for i, mp in enumerate(mask_paths):

    subject_dict[f"mask{i}"] = tio.LabelMap(mp)

base_subject = tio.Subject(**subject_dict)

# IMPORTANT:
# small pad instead of gigantic cube
if PAD > 0:

    base_subject = tio.Pad(PAD, padding_mode=-1000)(base_subject)


# ============================================================
# PREGENERATE RANDOM SO(3)
# ============================================================

def generate_inverted_rotations(num_samples: int) -> np.ndarray:
    """
    Генерирует матрицы поворота с равномерным распределением осей (SO(3)),
    но инвертированным распределением углов (смещение к малым углам).
    
    Возвращает:
    np.ndarray формы (num_samples, 3, 3) - массив матриц поворота
    """
    # 1. Генерируем стандартное равномерное распределение по мере Хаара
    r_original = R.random(num_samples, random_state = RNG_SEED)
    rotvecs = r_original.as_rotvec()
    
    # 2. Вычисляем текущие углы в радианах (длина векторов)
    angles_rad = np.linalg.norm(rotvecs, axis=1)
    
    # 3. Извлекаем чистые единичные векторы осей 
    # (добавлена защита от деления на идеальный ноль, 1e-10)
    safe_angles = np.where(angles_rad == 0, 1e-10, angles_rad)
    axes = rotvecs / safe_angles[:, np.newaxis]
    
    # 4. ПРИМЕНЯЕМ ИНВЕРСИЮ: 180 градусов (pi) минус исходный угол
    new_angles_rad = np.pi - angles_rad
    
    # 5. Собираем новые векторы вращения: умножаем оси на новые углы
    new_rotvecs = axes * new_angles_rad[:, np.newaxis]
    
    # 6. Превращаем обратно в классические матрицы 3x3
    matrices_3x3 = R.from_rotvec(new_rotvecs).as_matrix()
    
    return matrices_3x3

def generate_peak_rotation(num_samples: int, fraction: float = 0.5) -> np.ndarray:
    """
    Генерирует повороты на основе R.random(), искусственно смещая пик углов.
    fraction=0.5 смещает пик на 90 градусов.
    fraction=0.666 (2/3) смещает пик на 120 градусов.
    """
    # 1. Базовая честная генерация
    r_original = R.random(num_samples)
    rotvecs = r_original.as_rotvec()
    
    # 2. Извлекаем оси и углы
    angles_rad = np.linalg.norm(rotvecs, axis=1)
    safe_angles = np.where(angles_rad == 0, 1e-10, angles_rad)
    axes = rotvecs / safe_angles[:, np.newaxis]
    
    # 3. Сжимаем углы на заданную долю (если 0.5, то пик на 90°)
    compressed_angles = angles_rad * fraction
    
    # 4. Если мы сжали к 90 (fraction=0.5), делаем симметричное отражение,
    # чтобы заполнить правую половину пространства (от 90 до 180)
    if fraction == 0.5:
        # Случайная маска для 50% выборки
        flip_mask = np.random.rand(num_samples) > 0.5
        # Отражаем выбранную половину
        compressed_angles[flip_mask] = np.pi - compressed_angles[flip_mask]
    
    # 5. Собираем новые матрицы
    new_rotvecs = axes * compressed_angles[:, np.newaxis]
    matrices_3x3 = R.from_rotvec(new_rotvecs).as_matrix()
    
    return matrices_3x3

# ==========================================
# ПРИМЕР ИСПОЛЬЗОВАНИЯ В ПАЙПЛАЙНЕ
# ==========================================

# 1. Генерируем, например, 300 ракурсов малых углов (имитация вибраций конвейера)
#rotations = generate_peak_rotation(120)
#rotations = R.from_matrix(rotations)
rng = np.random.default_rng(RNG_SEED)

rotations = R.random(
    N_ROTATIONS,
    random_state=rng
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ROTATION_RUNS = []

# ============================================================
# GENERATE ROTATED DATA
# ============================================================

for idx, rot in enumerate(rotations):

    print(f"[{idx+1}/{N_ROTATIONS}] rotation")

    # ========================================================
    # representations
    # ========================================================

    degrees = rot.as_euler(
        "xyz",
        degrees=True
    )

    quat = rot.as_quat()

    matrix = rot.as_matrix()

    rotvec = rot.as_rotvec()

    angle_rad = np.linalg.norm(rotvec)

    angle_deg = float(
        np.rad2deg(angle_rad)
    )

    if angle_rad < 1e-8:

        axis = np.array(
            [0.0, 0.0, 1.0],
            dtype=float
        )

    else:

        axis = rotvec / angle_rad

    # ========================================================
    # affine transform
    # ========================================================

    affine = tio.Affine(

        scales=1,

        degrees=tuple(float(x) for x in degrees),

        translation=0,

        center="image",

        image_interpolation="linear",

        label_interpolation="nearest",

        default_pad_value="minimum",
    )

    # IMPORTANT:
    # rotate ORIGINAL image every time
    current = affine(base_subject)

    if CROP_BACK:

        current = tio.CropOrPad(orig_shape)(current)

    # ========================================================
    # output dirs
    # ========================================================

    run_dir = OUT_DIR / f"rot_{idx:03d}"

    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # ========================================================
    # save image (КТ-снимок)
    # ========================================================

    out_img = run_dir / "ct_rot.nii"
    
    # 1. Сохраняем КТ-снимок базовым методом
    current.img.save(out_img)

    # 2. Мгновенная оптимизация КТ-снимка (возвращаем к int16)
    # Это сожмет файл в 2 раза и спасет от MemoryError
    ct_nii = nib.load(out_img)
    ct_data = np.round(ct_nii.get_fdata()).astype(np.int16)
    nib.save(nib.Nifti1Image(ct_data, ct_nii.affine), out_img)


    # ========================================================
    # save masks
    # ========================================================

    out_masks = []

    for mi, mp in enumerate(mask_paths):

        out_m = run_dir / out_name(mp)

        # 1. Сохраняем маску базовым методом
        getattr(
            current,
            f"mask{mi}"
        ).save(out_m)

        out_masks.append(out_m)

        # 2. Мгновенная бинаризация и сжатие маски (в uint8)
        # Отсекаем артефакты интерполяции (float32)
        mask_nii = nib.load(out_m)
        clean_mask_data = (mask_nii.get_fdata() >= 0.5).astype(np.uint8)
        nib.save(nib.Nifti1Image(clean_mask_data, mask_nii.affine), out_m)

    # ========================================================
    # metadata
    # ========================================================

    meta = {

        "rotation_angle_deg":
            angle_deg,

        "rotation_axis_xyz":
            axis.tolist(),

        "degrees_xyz":
            degrees.tolist(),

        "quaternion_xyzw":
            quat.tolist(),

        "rotation_matrix":
            matrix.tolist(),
    }

    with open(
        run_dir / "rotation.json",
        "w"
    ) as f:

        json.dump(
            meta,
            f,
            indent=2
        )

    ROTATION_RUNS.append({

        "idx":
            idx,

        "rotation_angle_deg":
            angle_deg,

        "rotation_axis_xyz":
            axis.tolist(),

        "degrees_xyz":
            degrees.tolist(),

        "quaternion_xyzw":
            quat.tolist(),

        "rotation_matrix":
            matrix.tolist(),

        "dir":
            run_dir,

        "image":
            out_img,

        "masks":
            out_masks,
    })

    # ========================================================
    # MEMORY CLEANUP
    # ========================================================

    del current
    del affine

    gc.collect()


print(
    f"\nGenerated rotations: "
    f"{len(ROTATION_RUNS)}"
)



[1/300] rotation
[2/300] rotation
[3/300] rotation
[4/300] rotation
[5/300] rotation
[6/300] rotation
[7/300] rotation
[8/300] rotation
[9/300] rotation
[10/300] rotation
[11/300] rotation
[12/300] rotation
[13/300] rotation
[14/300] rotation
[15/300] rotation
[16/300] rotation
[17/300] rotation
[18/300] rotation
[19/300] rotation
[20/300] rotation
[21/300] rotation
[22/300] rotation
[23/300] rotation
[24/300] rotation
[25/300] rotation
[26/300] rotation
[27/300] rotation
[28/300] rotation
[29/300] rotation
[30/300] rotation
[31/300] rotation
[32/300] rotation
[33/300] rotation
[34/300] rotation
[35/300] rotation
[36/300] rotation
[37/300] rotation
[38/300] rotation
[39/300] rotation
[40/300] rotation
[41/300] rotation
[42/300] rotation
[43/300] rotation
[44/300] rotation
[45/300] rotation
[46/300] rotation
[47/300] rotation
[48/300] rotation
[49/300] rotation
[50/300] rotation
[51/300] rotation
[52/300] rotation
[53/300] rotation
[54/300] rotation
[55/300] rotation
[56/300] rotation
[

In [2]:
from pathlib import Path
OUT_DIR = Path(
    r"C:\Users\semen\SEprojects\segmentations\so3_benchmark"
)

OUT_DIR = Path(
    r"C:\Users\semen\SEprojects\segmentations\so3_benchmark"
)

In [2]:
import json
import pandas as pd
#если выходная директория другая - обычно с масками


rows = []
for rot_dir in sorted(OUT_DIR.glob("rot_*")):
    json_path = rot_dir / "rotation.json"
    if not json_path.exists():
        continue
    with open(json_path) as f:
        meta = json.load(f)
    rows.append({
        "rotation": rot_dir.name,
        **meta
    })

df_angles = pd.DataFrame(rows)
df_angles.to_csv(OUT_DIR / "rotations.csv", index=False)
df_angles

,rotation,rotation_angle_deg,rotation_axis_xyz,degrees_xyz,quaternion_xyzw,rotation_matrix
0,rot_000,147.440335,"[0.5582139791155529, -0.07585057131090701, 0.8...","[24.009805921343947, -62.967843108096964, 126....","[-0.5358318452262898, 0.07280926868113825, -0....","[[-0.268599940755993, -0.5226815481705845, 0.8..."
1,rot_001,145.963193,"[-0.17297130702543326, -0.9554317924731266, 0....","[-144.5863973134581, -27.329616762823715, 150....","[-0.16539703114494558, -0.9135941946331781, 0....","[[-0.7739658162651968, 0.16830840719273657, -0..."
2,rot_002,74.563091,"[0.28246220321954546, -0.3569913436168314, -0....","[30.802706714722728, -9.181216918979025, -70.7...","[-0.17109643907291988, 0.2162411358988878, 0.5...","[[0.3247250956332239, 0.7842602917460058, -0.5..."
3,rot_003,173.143679,"[-0.9906350954570587, -0.1364674493235325, -0....","[-173.27403920728054, -1.4232785809611213, 15....","[-0.9888624123117675, -0.1362232488622263, -0....","[[0.9628491011284129, 0.26992893386816874, -0...."
4,rot_004,156.323015,"[0.029310748971439787, -0.72500853173164, -0.6...","[90.51367035908162, -14.62598748461397, -160.8...","[0.028687302993003743, -0.7095874432469909, -0...","[[-0.9141780552418993, 0.23562124102802348, -0..."
...,...,...,...,...,...,...
535,rot_595,145.119337,"[0.14867957164700024, 0.5385845016500652, 0.82...","[64.32657442954853, 4.791950754884294, 141.521...","[0.14184471106547425, 0.5138255523245402, 0.79...","[[-0.7801050804099066, -0.32851261413922117, 0..."
536,rot_596,165.655061,"[-0.560235203386255, 0.7076502157875489, -0.43...","[-129.26680734607578, -17.431427385766046, -11...","[-0.5558512558534974, 0.7021127176103442, -0.4...","[[-0.3508804655260462, -0.6738694614385543, 0...."
537,rot_597,55.998656,"[-0.8775328723756687, -0.404599403114043, 0.25...","[-52.73568642948558, -13.643123312257291, 22.3...","[0.41196764210909115, 0.1899437244423615, -0.1...","[[0.8986470251186454, -0.056858348403690584, -..."
538,rot_598,168.426772,"[0.4617399667571729, 0.26463805454431255, -0.8...","[-38.618276432818355, 55.78967128776601, 172.6...","[0.4593870744347034, 0.26328953613222106, -0.8...","[[-0.5575961297519165, 0.41175238105765677, -0..."


In [3]:
# Ячейка 1 — записываем worker.py на диск
worker_code = '''
import sys, json, os, contextlib, time, inspect
from pathlib import Path
import numpy as np
import nibabel as nib

def roi_key(name):
    low = name.lower()
    for suf in (".nii.gz", ".nii"):
        if low.endswith(suf):
            name = name[:-len(suf)]; break
    if name.endswith("_rot"):
        name = name[:-4]
    return name.lower()

def load_mask(path):
    nii = nib.load(str(path))
    data = np.asanyarray(nii.dataobj, dtype=np.uint8) > 0
    nii.uncache()
    return data

def seg_metrics(gt, pr):
    tp = np.logical_and(gt, pr).sum()
    fp = np.logical_and(~gt, pr).sum()
    fn = np.logical_and(gt, ~pr).sum()
    s = gt.sum() + pr.sum()
    if s == 0:
        return {"dice":1.0,"iou":1.0,"precision":1.0,"recall":1.0}
    return {
        "dice":      float(2*tp / s),
        "iou":       float(tp / (tp+fp+fn)),
        "precision": float(tp/(tp+fp)) if (tp+fp)>0 else float("nan"),
        "recall":    float(tp/(tp+fn)) if (tp+fn)>0 else float("nan"),
    }

if __name__ == "__main__":
    args = json.loads(sys.argv[1])
    run_dir    = Path(args["run_dir"])
    roi_subset = args["roi_subset"]

    from totalsegmentator.python_api import totalsegmentator

    in_img = run_dir / "ct_rot.nii"
    if not in_img.exists():
        in_img = run_dir / "ct_rot.nii.gz"

    gt_masks = {
        roi_key(p.name): p
        for p in run_dir.glob("*_rot.nii*")
        if not p.name.startswith("ct_rot")
    }

    ts_out_dir = run_dir / "totalseg_rois"
    ts_out_dir.mkdir(exist_ok=True)

    sig = inspect.signature(totalsegmentator)
    kwargs = dict(input=in_img, output=ts_out_dir,
                  task=args["ts_task"], fast=args["ts_fast"],
                  quiet=True, body_seg=False, verbose=False)
    if "roi_subset" in sig.parameters:
        kwargs["roi_subset"] = roi_subset

    start = time.time()
    with open(os.devnull,"w") as dn, \\
         contextlib.redirect_stdout(dn), contextlib.redirect_stderr(dn):
        totalsegmentator(**kwargs)
    elapsed = time.time() - start

    results = []
    for mf in sorted(ts_out_dir.glob("*.nii*")):
        roi = roi_key(mf.name)
        if roi not in roi_subset:
            continue
        gt_path = gt_masks.get(roi)
        if gt_path is None:
            continue
        gt = load_mask(gt_path)
        pr = load_mask(mf)
        m  = seg_metrics(gt, pr)
        del gt, pr
        results.append({"roi": roi, "time_s": elapsed, **m})

    print(json.dumps(results))
'''

with open("worker.py", "w", encoding="utf-8") as f:
    f.write(worker_code)

print("worker.py saved")

worker.py saved


In [4]:
main_code = f"""
import ctypes
ctypes.windll.kernel32.SetThreadExecutionState(0x80000003)

import json, subprocess, sys
from pathlib import Path
import pandas as pd

# Теперь скрипт сам подставит путь из TARGET_DIRECTORY
ROOT_DIR = Path(r"{OUT_DIR}")
ROI_SUBSET = ["liver", "stomach", "spleen", "pancreas", "kidney_right", "kidney_left", "gallbladder", "hip_left", "hip_right", "sacrum"]
TS_TASK    = "total"
TS_FAST    = True

rotation_dirs = sorted([p for p in ROOT_DIR.glob("rot_*") if p.is_dir()])
print("Found rotations:", len(rotation_dirs))

rows = []

for idx, run_dir in enumerate(rotation_dirs):
    if idx > 300:
        continue
    print(f"[{{idx+1}}/{{len(rotation_dirs)}}] {{run_dir.name}} ...", flush=True)

    payload = json.dumps({{
        "run_dir":    str(run_dir),
        "roi_subset": ROI_SUBSET,
        "ts_task":    TS_TASK,
        "ts_fast":    TS_FAST,
    }})

    try:
        proc = subprocess.Popen(
            [sys.executable, "worker.py", payload],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            creationflags=subprocess.CREATE_NO_WINDOW,
        )
        stdout, stderr = proc.communicate(timeout=600)
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"TIMEOUT на {{run_dir.name}}")
        continue

    if proc.returncode != 0:
        print(f"FAILED: {{stderr[-500:]}}")
        continue

    try:
        last_line = stdout.strip().splitlines()[-1]
        entries = json.loads(last_line)
    except Exception as e:
        print(f"PARSE ERROR: {{e}}")
        continue

    for entry in entries:
        rows.append({{"rotation": run_dir.name, **entry}})

    print(f"OK — {{len(entries)}} ROIs", flush=True)

    df = pd.DataFrame(rows) 
    df.to_csv(ROOT_DIR / "results.csv", index=False)
    print("Saved results.csv")
"""

with open("main.py", "w", encoding="utf-8") as f:
    f.write(main_code)

print("main.py saved")

main.py saved


In [ ]:
# Ячейка 2 — запускаем через PowerShell и читаем результат
import subprocess, sys, ctypes

# запрещаем сон прямо из Python
ctypes.windll.kernel32.SetThreadExecutionState(0x80000003)

proc = subprocess.Popen(
    [sys.executable, "main.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    encoding="utf-8",
    errors="replace",
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()

Found rotations: 540
[1/540] rot_000 ...
OK — 10 ROIs
Saved results.csv
[2/540] rot_001 ...
OK — 10 ROIs
Saved results.csv
[3/540] rot_002 ...
OK — 10 ROIs
Saved results.csv
[4/540] rot_003 ...
OK — 10 ROIs
Saved results.csv
[5/540] rot_004 ...
OK — 10 ROIs
Saved results.csv
[6/540] rot_005 ...
OK — 10 ROIs
Saved results.csv
[7/540] rot_006 ...
OK — 10 ROIs
Saved results.csv
[8/540] rot_007 ...
OK — 10 ROIs
Saved results.csv
[9/540] rot_008 ...
OK — 10 ROIs
Saved results.csv
[10/540] rot_009 ...
OK — 10 ROIs
Saved results.csv
[11/540] rot_010 ...
OK — 10 ROIs
Saved results.csv
[12/540] rot_011 ...
OK — 10 ROIs
Saved results.csv
[13/540] rot_012 ...
OK — 10 ROIs
Saved results.csv
[14/540] rot_013 ...
OK — 10 ROIs
Saved results.csv
[15/540] rot_014 ...
OK — 10 ROIs
Saved results.csv
[16/540] rot_015 ...
OK — 10 ROIs
Saved results.csv
[17/540] rot_016 ...
OK — 10 ROIs
Saved results.csv
[18/540] rot_017 ...
OK — 10 ROIs
Saved results.csv
[19/540] rot_018 ...
OK — 10 ROIs
Saved results.csv


In [ ]:
import nibabel as nib

In [ ]:
print("rot_000 type:", nib.load(OUT_DIR / "rot_000" / "ct_rot.nii").get_data_dtype())

# Проверяем файл, на котором всё сломалось
print("rot_001 type:", nib.load(OUT_DIR / "rot_001" / "ct_rot.nii").get_data_dtype())

In [ ]:
import pandas as pd
import numpy as np
import nibabel as nib
from pathlib import Path

# Путь к папке, где лежат ваши rot_000, rot_001 и т.д.
ROOT_DIR = Path(r"C:\Users\semen\SEprojects\segmentations\so3_benchmark") 

def main():
    base_mask_path = Path(
        r"C:\Users\semen\SEprojects\segmentations\unzip_seg\liver.nii"
    )
    if not base_mask_path.exists():
        print(f"Ошибка: Не найден эталонный файл {base_mask_path}")
        return

    # Считаем исходный объем эталона
    v_base = np.sum(nib.load(base_mask_path).get_fdata() > 0)
    
    rows = []
    rotation_dirs = sorted([p for p in ROOT_DIR.glob("rot_*") if p.is_dir()])
    
    for run_dir in rotation_dirs:
        mask_path = run_dir / "liver_rot.nii"
        if not mask_path.exists():
            continue
            
        v_curr = np.sum(nib.load(mask_path).get_fdata() > 0)
        loss_pct = max(0.0, ((v_base - v_curr) / v_base) * 100.0)
        
        rows.append({
            "rotation": run_dir.name,
            "loss_percent": round(loss_pct, 2)
        })
        print(f"{run_dir.name}: {round(loss_pct, 2)}%")

    # Сохраняем строго две колонки
    df = pd.DataFrame(rows)
    df.to_csv(ROOT_DIR / "crop_losses.csv", index=False)
    print("\nРезультаты записаны в crop_losses.csv")

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import nibabel as nib
from pathlib import Path

def sanitize_nifti_types(root_dir="."):
    root_path = Path(root_dir)
    rotation_dirs = sorted([p for p in root_path.glob("rot_*") if p.is_dir()])
    
    print(f"Найдено папок: {len(rotation_dirs)}. Начинаем конвертацию...")
    
    for i, rot_dir in enumerate(rotation_dirs):
        vol_path = rot_dir / "ct_rot.nii"
        mask_path = rot_dir / "liver_rot.nii"
        
        # 1. Лечим КТ-снимок (volume) -> int16
        if vol_path.exists():
            vol_img = nib.load(vol_path)
            # Округляем (чтобы не потерять данные) и жестко конвертируем
            vol_data = np.round(vol_img.get_fdata()).astype(np.int16)
            # Сохраняем поверх старого файла
            nib.save(nib.Nifti1Image(vol_data, vol_img.affine), vol_path)
            
        # 2. Лечим Маску (mask) -> uint8
        if mask_path.exists():
            mask_img = nib.load(mask_path)
            # Для масок используем порог 0.5, чтобы вернуть дробные значения к 0 или 1
            mask_data = (mask_img.get_fdata() >= 0.5).astype(np.uint8)
            nib.save(nib.Nifti1Image(mask_data, mask_img.affine), mask_path)
            
        print(f"[{i+1}/{len(rotation_dirs)}] {rot_dir.name} исправлен (int16/uint8).")

    print("\nУСПЕХ! Все файлы конвертированы. Теперь nnU-Net должен их проглотить.")

if __name__ == "__main__":
    sanitize_nifti_types(OUT_DIR)

In [4]:
import os
import glob
import re
import gc
import numpy as np
import nibabel as nib
import pandas as pd

# 1. ОСНОВНЫЕ НАСТРОЙКИ
BASE_DIR = OUT_DIR  # Папка, внутри которой лежат rot_0, rot_1 и т.д.
START_ROTATION = 300                     # Начать обработку со 126-й папки
OUTPUT_CSV = "foreground_iou_results_bones.csv"

# 2. ГИБКАЯ НАСТРОЙКА КЛАССОВ (Оставляем только почки, как вы просили)
# Если захотите вернуть все органы, просто добавьте 'liver', 'spleen' и т.д.
TARGET_ROIS = [
    'sacrum',
    'hip_left',
    'hip_right',
]

results = []

# Вспомогательная функция для извлечения номера ротации из названия папки (например, "rot_126" -> 126)
def get_rot_index(folder_name):
    match = re.search(r'\d+', folder_name)
    return int(match.group()) if match else -1

print(f"Сканирование директории: {BASE_DIR}")
# Находим все папки rot_*
all_rot_folders = [f for f in glob.glob(os.path.join(BASE_DIR, "rot_*")) if os.path.isdir(f)]
# Сортируем их по номеру, чтобы идти по порядку
all_rot_folders.sort(key=lambda x: get_rot_index(os.path.basename(x)))

# 3. ОСНОВНОЙ ЦИКЛ ОБРАБОТКИ
for folder in all_rot_folders:
    folder_name = os.path.basename(folder)
    rot_idx = get_rot_index(folder_name)
    
    # Пропускаем все папки до 126
    if rot_idx > 261:
        continue
        
    print(f"[{folder_name}] Обработка...")
    
    totalseg_dir = os.path.join(folder, "totalseg_rois")
    if not os.path.exists(totalseg_dir):
        print(f"  -> Пропуск: папка предсказаний {totalseg_dir} не найдена.")
        continue

    combined_gt = None
    combined_pred = None
    found_any_roi = False
    
    for roi in TARGET_ROIS:
        # ИЩЕМ GROUND TRUTH:
        # Лежит в корне folder, содержит имя roi, и НЕ начинается с 'ct_rot'
        gt_pattern = os.path.join(folder, f"*{roi}*.nii")
        gt_candidates = [f for f in glob.glob(gt_pattern) if not os.path.basename(f).startswith("ct_rot")]
        
        # ИЩЕМ ПРЕДСКАЗАНИЕ:
        # Лежит в подпапке totalseg_rois
        pred_pattern = os.path.join(totalseg_dir, f"*{roi}*.nii.gz")
        pred_candidates = glob.glob(pred_pattern)
        
        if gt_candidates and pred_candidates:
            try:
                # Пытаемся безопасно загрузить бинарные маски
                gt_data = nib.load(gt_candidates[0]).get_fdata() > 0
                pred_data = nib.load(pred_candidates[0]).get_fdata() > 0
                
                # Если это первый найденный орган в этой папке, создаем пустой холст
                if combined_gt is None:
                    combined_gt = np.zeros(gt_data.shape, dtype=bool)
                    combined_pred = np.zeros(pred_data.shape, dtype=bool)
                    
                # Склеиваем органы (Агностический подход)
                combined_gt = np.logical_or(combined_gt, gt_data)
                combined_pred = np.logical_or(combined_pred, pred_data)
                found_any_roi = True
                
            except EOFError:
                # Ловим конкретно ошибку недописанного архива
                print(f"  [!] КРИТИЧЕСКАЯ ОШИБКА ЧТЕНИЯ (БИТЫЙ ФАЙЛ): Пропуск {roi}")
                print(f"      Проверьте файлы: {gt_candidates[0]} \n      или {pred_candidates[0]}")
                continue # Пропускаем этот орган и идем к следующему
            except Exception as e:
                # Ловим любые другие сбои чтения
                print(f"  [!] НЕПРЕДВИДЕННАЯ ОШИБКА с {roi}: {e}")
                continue
        else:
            print(f"  -> Внимание: для {roi} не найдена пара GT/Pred, пропускаем орган.")

    # 4. РАСЧЕТ МЕТРИКИ ДЛЯ ПАПКИ
    if found_any_roi and combined_gt is not None:
        intersection = np.logical_and(combined_gt, combined_pred).sum()
        union = np.logical_or(combined_gt, combined_pred).sum()
        
        foreground_iou = intersection / union if union > 0 else 0.0
        
        results.append({
            'rotation_folder': folder_name,
            'rotation_index': rot_idx,
            'foreground_iou': foreground_iou
        })
        print(f"  -> Foreground IoU: {foreground_iou:.4f}")
        
    # Очистка памяти (критично для больших датасетов)
    del combined_gt, combined_pred
    gc.collect()

# 5. СОХРАНЕНИЕ В CSV
if results:
    df = pd.DataFrame(results)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nГотово! Результаты записаны в {OUTPUT_CSV}")
else:
    print("\nНет данных для сохранения.")

Сканирование директории: C:\Users\semen\SEprojects\segmentations\so3_benchmark
[rot_000] Обработка...
  -> Foreground IoU: 0.6744
[rot_001] Обработка...
  -> Foreground IoU: 0.7746
[rot_002] Обработка...
  -> Foreground IoU: 0.7617
[rot_003] Обработка...
  -> Foreground IoU: 0.7192
[rot_004] Обработка...
  -> Foreground IoU: 0.7481
[rot_005] Обработка...
  -> Foreground IoU: 0.7175
[rot_006] Обработка...
  -> Foreground IoU: 0.5837
[rot_007] Обработка...
  -> Foreground IoU: 0.6546
[rot_008] Обработка...
  -> Foreground IoU: 0.6988
[rot_009] Обработка...
  -> Foreground IoU: 0.7412
[rot_010] Обработка...
  -> Foreground IoU: 0.7447
[rot_011] Обработка...
  -> Foreground IoU: 0.5435
[rot_012] Обработка...
  -> Foreground IoU: 0.8106
[rot_013] Обработка...
  -> Foreground IoU: 0.6908
[rot_014] Обработка...
  -> Foreground IoU: 0.6920
[rot_015] Обработка...
  -> Foreground IoU: 0.8154
[rot_016] Обработка...
  -> Foreground IoU: 0.7454
[rot_017] Обработка...
  -> Foreground IoU: 0.7089
[ro